# FEC Candidate/Committee Alignment Tool

## Imports and Data Preparation

In [1]:
# imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import umap.umap_ as umap
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler

from sklearn.neighbors import NearestNeighbors

In [2]:
cand_summ_22 = pd.read_parquet('./datasets/cand_summ_22.parquet', engine="fastparquet")
cand_summ_24 = pd.read_parquet('./datasets/cand_summ_24.parquet', engine="fastparquet")

comm_summ_22 = pd.read_parquet('./datasets/comm_summ_22.parquet', engine="fastparquet")
comm_summ_24 = pd.read_parquet('./datasets/comm_summ_24.parquet', engine="fastparquet")

cc_ie_22 = pd.read_parquet('./datasets/cc_ie_22.parquet', engine="fastparquet")
cc_ie_24 = pd.read_parquet('./datasets/cc_ie_24.parquet', engine="fastparquet")

In [3]:
cc_ie_22 = cc_ie_22.dropna(subset='CAND_ID')

support_codes = {"24E", "24F", "24C", "24H", "24K" ,"24P","24R", "24Z"}
oppose_codes = {"24A", "24N"}

cc_ie_22 = cc_ie_22.copy()
cc_ie_22["support_oppose_flag"] = "N"

cc_ie_22.loc[cc_ie_22["TRANSACTION_TP"].isin(support_codes), "support_oppose_flag"] = "S"

cc_ie_22.loc[cc_ie_22["TRANSACTION_TP"].isin(oppose_codes), "support_oppose_flag"] = "O"

cc_ie_22 = cc_ie_22.sort_values(["CMTE_ID", "TRAN_ID", "FILE_NUM"])
cc_ie_22 = cc_ie_22.groupby(["CMTE_ID", "TRAN_ID"], as_index=False).last().copy()

In [4]:
cand_interaction_matrix = (
    cc_ie_22[cc_ie_22['support_oppose_flag']=='S'].pivot_table(
        index="CAND_ID",          # rows
        columns="CMTE_ID",        # columns
        values="TRANSACTION_AMT", # cell values
        aggfunc="sum",            # sum of transactions
        fill_value=0              # replace NaNs with 0
    )
)

candidate_totals = cand_interaction_matrix.sum(axis=1)
candidate_totals = candidate_totals.replace(0, 1)
cand_interaction_matrix_scaled = cand_interaction_matrix.div(candidate_totals, axis=0)

In [5]:
comm_interaction_matrix = (
    cc_ie_22[cc_ie_22['support_oppose_flag']=='S'].pivot_table(
        index="CMTE_ID",          # rows
        columns="CAND_ID",        # columns
        values="TRANSACTION_AMT", # cell values
        aggfunc="sum",            # sum of transactions
        fill_value=0              # replace NaNs with 0
    )
)

committee_totals = comm_interaction_matrix.sum(axis=1)
committee_totals = committee_totals.replace(0, 1)
comm_interaction_matrix_scaled = comm_interaction_matrix.div(committee_totals, axis=0)

### Latent Feature Matrix Construction

In [6]:
# pivot table creation, and feature engineering!!

#30 #20
svd_cand = TruncatedSVD(n_components=15, random_state=56)
scaler_cand = StandardScaler()
latent = svd_cand.fit_transform(cand_interaction_matrix_scaled)
scaler_latent = scaler_cand.fit_transform(latent)

cand_scaled_latent = pd.DataFrame(
    scaler_latent,
    index=cand_interaction_matrix_scaled.index,
    columns=[f"latent_{i+1}" for i in range(scaler_latent.shape[1])]
)

svd_comm = TruncatedSVD(n_components=15, random_state=56)
scaler_comm = StandardScaler()
latent = svd_comm.fit_transform(comm_interaction_matrix_scaled)
scaler_latent = scaler_comm.fit_transform(latent)

comm_scaled_latent = pd.DataFrame(
    scaler_latent,
    index=comm_interaction_matrix_scaled.index,
    columns=[f"latent_{i+1}" for i in range(scaler_latent.shape[1])]
)

In [174]:
# construct data frame of incumbents in the datasets

latent_incumb_22 = pd.merge(cand_summ_22[['Cand_Id','Cand_Incumbent_Challenger_Open_Seat']],cand_scaled_latent, left_on='Cand_Id', right_on='CAND_ID')
#latent_incumb_22 = cand_summ_22[['Cand_Id','Cand_Incumbent_Challenger_Open_Seat']].copy()
latent_incumb_22['IS_INCUMB'] = latent_incumb_22['Cand_Incumbent_Challenger_Open_Seat'] == 'INCUMBENT'
latent_incumb_22 = latent_incumb_22.drop(columns='Cand_Incumbent_Challenger_Open_Seat').copy()
latent_incumb_22.set_index('Cand_Id', inplace=True)
incumbs_df = latent_incumb_22[['IS_INCUMB']].copy()

## Alignment Tool Functions

In [42]:
#accepts a matrix and a contribution distribution and returns the scaled point vector and matrix
def build_persona(dist, matrix, scaler,svd):
    cols = matrix.columns
    matrix_copy = matrix.copy()
    data = {c: [0.0] for c in cols}

    persona_point = pd.DataFrame(data)

    ##Add data
    for cat, prct in dist.items():
        if cat in persona_point.columns:
            persona_point.loc[0, cat] = prct

    matrix_copy.loc[0] = persona_point.iloc[0]
    
    svd = TruncatedSVD(n_components=15, random_state=56)
    scaler = StandardScaler()
    latent = svd.fit_transform(matrix_copy)
    scaled_latent = scaler.fit_transform(latent)

    scaled_latent_df = pd.DataFrame(
        scaled_latent,
        index= matrix_copy.index,
        columns=[f"latent_{i+1}" for i in range(scaled_latent.shape[1])])

    persona_point_latent = scaled_latent_df.loc[0]
    
    return persona_point_latent, scaled_latent_df
    

In [138]:
def find_neighbors_cand(vector, matrix,incumbs):

    matrix_incumbs_tf = matrix.join(incumbs)
    matrix_incumbs_tf = matrix_incumbs_tf.fillna(False)
    
    matrix_incumbs = matrix_incumbs_tf[matrix_incumbs_tf['IS_INCUMB']].copy()

    #display(matrix_incumbs)

    matrix_incumbs.drop('IS_INCUMB', axis=1, inplace=True)
    
    knn = NearestNeighbors(n_neighbors=5, metric='euclidean')
    knn.fit(matrix_incumbs)
    
    distances, indices = knn.kneighbors([vector])
    
    n_indices = matrix_incumbs.index[indices[0]]
    
    print("Neighbor indices:", matrix_incumbs.index[indices[0]])
    print("Distances:", distances[0])
    
    for i in n_indices:
        display(cand_summ_22[cand_summ_22['Cand_Id'] == i][['Cand_Name', 'Cand_Id', 'Cand_Office', 'Cand_Office_St',
           'Cand_Office_Dist', 'Cand_Party_Affiliation',
           'Cand_Incumbent_Challenger_Open_Seat']])


In [131]:
def find_neighbors_comm(vector, matrix):

    #display(matrix_incumbs)
    
    knn = NearestNeighbors(n_neighbors=10, metric='euclidean')
    knn.fit(matrix)
    
    distances, indices = knn.kneighbors([vector])
    
    n_indices = matrix.index[indices[0]]
    
    print("Neighbor indices:", matrix.index[indices[0]])
    print("Distances:", distances[0])
    
    for i in n_indices:
        display(comm_summ_22[comm_summ_22['CMTE_ID'] == i][['CMTE_ID','CMTE_NM','CMTE_TP','CMTE_DSGN','CMTE_CITY','CMTE_ST']])

## Alignment Tool Use

In [171]:
vector, matrix = build_persona({'C00135368': 0.0, #SIERRA CLUB POLITICAL COMMITTEE
                                'C00806893': 0.0, #JANE FONDA CLIMATE PAC
                                'C00035006': 0.0, #CHEVRON EMPLOYEES POLITICAL ACTION COMMITTEE
                                'C00330233': 0.0, #ALLIANCE COAL, LLC PAC
                                'C00330233': 0.0, #KOCH, INC. POLITICAL ACTION COMMITTEE 
                                'C00093054': 10.0, #WALMART INC. PAC FOR RESPONSIBLE GOVERNMENT
                                'C00098061': 10.0, #TARGET CORPORATION CITIZENS POLITICAL FORUM
                                'C00284885': 10.0, #THE HOME DEPOT INC. POLITICAL ACTION COMMITTEE
                                'C00640086': 0.0, #EVERYTOWN FOR GUN SAFETY ACTION FUND, INC. PAC (EVERYTOWN PAC)
                                'C00053553': 0.0 #NATIONAL RIFLE ASSOCIATION OF AMERICA POLITICAL VICTORY FUND
                               },
              cand_interaction_matrix_scaled,
              scaler_cand,
              svd_cand)
#display(vector)
#display(matrix)

find_neighbors_cand(vector, matrix, incumbs_df)


Neighbor indices: Index(['H6MD07020', 'H0GA09246', 'H2VI00082', 'H2GA12121', 'H0AR03055'], dtype='object', name='CAND_ID')
Distances: [46.30552275 46.32284347 46.32455345 46.32650792 46.3267688 ]


C:\Users\wkmp2\AppData\Local\Temp\ipykernel_28740\1033256873.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  matrix_incumbs_tf = matrix_incumbs_tf.fillna(False)
C:\Users\wkmp2\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but NearestNeighbors was fitted with feature names
  warnings.warn(


,Cand_Name,Cand_Id,Cand_Office,Cand_Office_St,Cand_Office_Dist,Cand_Party_Affiliation,Cand_Incumbent_Challenger_Open_Seat
3384,"MFUME, KWEISI",H6MD07020,H,MD,7.0,DEM,INCUMBENT


,Cand_Name,Cand_Id,Cand_Office,Cand_Office_St,Cand_Office_Dist,Cand_Party_Affiliation,Cand_Incumbent_Challenger_Open_Seat
2305,"CLYDE, ANDREW",H0GA09246,H,GA,9.0,REP,INCUMBENT


,Cand_Name,Cand_Id,Cand_Office,Cand_Office_St,Cand_Office_Dist,Cand_Party_Affiliation,Cand_Incumbent_Challenger_Open_Seat
3279,"PLASKETT, STACEY",H2VI00082,H,VI,0.0,DEM,INCUMBENT


,Cand_Name,Cand_Id,Cand_Office,Cand_Office_St,Cand_Office_Dist,Cand_Party_Affiliation,Cand_Incumbent_Challenger_Open_Seat
1666,"ALLEN, RICHARD W",H2GA12121,H,GA,12.0,REP,INCUMBENT


,Cand_Name,Cand_Id,Cand_Office,Cand_Office_St,Cand_Office_Dist,Cand_Party_Affiliation,Cand_Incumbent_Challenger_Open_Seat
4029,"WOMACK, STEVE",H0AR03055,H,AR,3.0,REP,INCUMBENT


In [191]:
vector, matrix = build_persona({'H8NY15148':0.0, #OCASIO-CORTEZ, ALEXANDRIA
                                'H0GA06192': 0.0, #GREENE, MARJORIE TAYLOR
                                'S6PA00274': 10.0, #FETTERMAN, JOHN KARL
                                'S2PA00638': 10.0 #OZ, MEHMET DR
                               },
              comm_interaction_matrix_scaled,
              scaler_comm,
              svd_comm)
#display(vector)
#display(matrix)

find_neighbors_comm(vector, matrix)

Neighbor indices: Index([          0, 'C90021635', 'C00637371', 'C00612846', 'C90021932',
       'C90021684', 'C00713693', 'C90021643', 'C00760801', 'C90021999'],
      dtype='object', name='CMTE_ID')
Distances: [ 0.         66.25466472 66.25466472 66.25466472 66.25466472 66.25466472
 66.25466472 66.25466472 66.25466472 66.25466472]


C:\Users\wkmp2\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but NearestNeighbors was fitted with feature names
  warnings.warn(


,CMTE_ID,CMTE_NM,CMTE_TP,CMTE_DSGN,CMTE_CITY,CMTE_ST


,CMTE_ID,CMTE_NM,CMTE_TP,CMTE_DSGN,CMTE_CITY,CMTE_ST
7498,C90021635,UFCW LOCAL 1776,I,U,PLYMOUTH MEETING,PA


,CMTE_ID,CMTE_NM,CMTE_TP,CMTE_DSGN,CMTE_CITY,CMTE_ST
2514,C00637371,CHRISSY HOULAHAN FOR CONGRESS,H,P,DEVON,PA


...

,CMTE_ID,CMTE_NM,CMTE_TP,CMTE_DSGN,CMTE_CITY,CMTE_ST
4655,C90021684,PA STANDS UP STATE PAC,I,U,LANCASTER,PA


,CMTE_ID,CMTE_NM,CMTE_TP,CMTE_DSGN,CMTE_CITY,CMTE_ST
11795,C00713693,FINELLO PA PAC,H,P,PHILADELPHIA,PA


,CMTE_ID,CMTE_NM,CMTE_TP,CMTE_DSGN,CMTE_CITY,CMTE_ST
10981,C90021643,PA STANDS UP POWER,I,U,LANCASTER,PA


,CMTE_ID,CMTE_NM,CMTE_TP,CMTE_DSGN,CMTE_CITY,CMTE_ST
4512,C00760801,THE TURNOUT PROJECT,O,U,WASHINGTON,DC


,CMTE_ID,CMTE_NM,CMTE_TP,CMTE_DSGN,CMTE_CITY,CMTE_ST
9121,C90021999,PHILLY NEIGHBORHOOD NETWORKS,I,U,PHILADELPHIA,PA


### Top contributors/candidates (for ad hoc validation of alignment results)

In [195]:
comm_cand_totals = cc_ie_22.groupby(['CMTE_ID','CAND_ID'])['TRANSACTION_AMT'].sum().reset_index().sort_values('TRANSACTION_AMT', ascending=False).copy()

# display a list of the top ten committee contributors of a speicified candidate
def display_top_donors(cand_id):
    display(comm_cand_totals[comm_cand_totals['CAND_ID'] == cand_id].head(10).merge(comm_summ_22[['CMTE_ID','CMTE_NM']], on='CMTE_ID').merge(cand_summ_22[['Cand_Id','Cand_Name']], left_on='CAND_ID', right_on='Cand_Id'))
    
# display a list of the top 10 candidates a committe has contributed to
def display_top_cmte_cands(cmte_id):
    display(comm_cand_totals[comm_cand_totals['CMTE_ID'] == cmte_id].head(10).merge(comm_summ_22[['CMTE_ID','CMTE_NM']], on='CMTE_ID').merge(cand_summ_22[['Cand_Id','Cand_Name']], left_on='CAND_ID', right_on='Cand_Id'))

In [196]:
display_top_donors('H2VI00082')

,CMTE_ID,CAND_ID,TRANSACTION_AMT,CMTE_NM,Cand_Id,Cand_Name
0,C00303339,H2VI00082,20000.0,NATIONAL ASSOCIATION OF REAL ESTATE INVESTMENT...,H2VI00082,"PLASKETT, STACEY"
1,C00130773,H2VI00082,10000.0,NATIONAL MULTIFAMILY HOUSING COUNCIL POLITICAL...,H2VI00082,"PLASKETT, STACEY"
2,C00007542,H2VI00082,10000.0,"INTERNATIONAL ASSOCIATION OF SHEET METAL, AIR,...",H2VI00082,"PLASKETT, STACEY"
3,C00284885,H2VI00082,10000.0,THE HOME DEPOT INC. POLITICAL ACTION COMMITTEE,H2VI00082,"PLASKETT, STACEY"
4,C00340075,H2VI00082,10000.0,BASF CORPORATION EMPLOYEES POLITICAL ACTION CO...,H2VI00082,"PLASKETT, STACEY"
5,C00028860,H2VI00082,10000.0,"AMERICAN FEDERATION OF TEACHERS, AFL-CIO COMMI...",H2VI00082,"PLASKETT, STACEY"
6,C00388827,H2VI00082,8500.0,ROCK HOLDINGS INC. PAC,H2VI00082,"PLASKETT, STACEY"
7,C00536573,H2VI00082,6000.0,ABBVIE POLITICAL ACTION COMMITTEE,H2VI00082,"PLASKETT, STACEY"
8,C00096156,H2VI00082,6000.0,HONEYWELL INTERNATIONAL POLITICAL ACTION COMMI...,H2VI00082,"PLASKETT, STACEY"
9,C00076299,H2VI00082,5500.0,CME GROUP INC. PAC,H2VI00082,"PLASKETT, STACEY"


In [197]:
display_top_cmte_cands('C90021635')

,CMTE_ID,CAND_ID,TRANSACTION_AMT,CMTE_NM,Cand_Id,Cand_Name
0,C90021635,S6PA00274,9629.0,UFCW LOCAL 1776,S6PA00274,"FETTERMAN, JOHN KARL"
